# LIBERO-Plus failure analysis (4 Joint models)

Compare **RGB**, **Depth**, **Flow**, and **Multimodal** Joint checkpoints on the same Plus task list.

**Kernel:** select `FastWAM (.venv)` (`.venv/bin/python`).

If packages are missing from the venv:
```bash
export UV_CACHE_DIR=/data/mgovind/.cache/uv
cd /data/mgovind/FastWAM
uv pip install --python .venv/bin/python ipykernel matplotlib jupyter
.venv/bin/python -m ipykernel install --user --name fastwam --display-name "FastWAM (.venv)"
```

In [ ]:
from __future__ import annotations

import json
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

REPO = Path("/data/mgovind/FastWAM").resolve()
EVAL_ROOT = REPO / "evaluate_results" / "libero_plus"

# Edit run timestamps if you re-evaluate.
RUNS = {
    "RGB": EVAL_ROOT / "libero_joint_2cam224_1e-4" / "20260831_144424",
    "Depth": EVAL_ROOT / "libero_joint_2cam224_1e-4_depth" / "20260831_102459",
    "Flow": EVAL_ROOT / "libero_joint_2cam224_1e-4_flow" / "20260831_123437",
    "MM": EVAL_ROOT / "libero_joint_2cam224_1e-4_rgb_depth_flow" / "20260831_164802",
}

MODELS = list(RUNS.keys())
CAT_SHORT = {
    "Camera Viewpoints": "Camera",
    "Background Textures": "Background",
    "Sensor Noise": "Noise",
    "Objects Layout": "Layout",
}

for name, run_dir in RUNS.items():
    summary = run_dir / "summary.json"
    if not summary.exists():
        raise FileNotFoundError(f"Missing {summary} for {name}")

print("Run directories OK:")
for name, run_dir in RUNS.items():
    print(f"  {name:5s} {run_dir}")

In [ ]:
def load_summary(run_dir: Path) -> dict:
    with open(run_dir / "summary.json", encoding="utf-8") as f:
        return json.load(f)


def build_task_frame(summaries: dict[str, dict]) -> pd.DataFrame:
    """One row per Plus task; columns include per-model success + metadata."""
    ref_model = MODELS[0]
    task_ids = sorted(summaries[ref_model]["task_results"].keys())
    rows = []
    for tid in task_ids:
        meta = summaries[ref_model]["task_results"][tid]
        row = {
            "task_id": tid,
            "suite": "_".join(tid.split("_")[:2]),
            "category": meta.get("perturbation_category"),
            "category_short": CAT_SHORT.get(meta.get("perturbation_category"), "?"),
            "difficulty": meta.get("difficulty_level"),
            "task_name": meta.get("task_name"),
            "description": meta.get("task_description"),
        }
        for m in MODELS:
            r = summaries[m]["task_results"][tid]
            row[f"success_{m}"] = bool(int(r["successes"]))
        rows.append(row)
    df = pd.DataFrame(rows)
    df["n_success"] = df[[f"success_{m}" for m in MODELS]].sum(axis=1)
    df["n_fail"] = len(MODELS) - df["n_success"]
    df["bucket"] = df["n_success"].map({4: "all_success", 0: "all_fail"}).fillna("mixed")
    return df


summaries = {m: load_summary(RUNS[m]) for m in MODELS}
tasks = build_task_frame(summaries)
print(f"Tasks: {len(tasks)}")
tasks.head()

## 1. Leaderboard & attribution (category means)

In [ ]:
def category_rates(summaries: dict[str, dict]) -> pd.DataFrame:
    cats = ["Camera", "Background", "Noise", "Layout"]
    rows = []
    for m in MODELS:
        lb = summaries[m]["plus_leaderboard"]
        row = {c: lb[c]["success_rate"] for c in cats}
        row["Total"] = lb["Total"]["success_rate"]
        row["model"] = m
        rows.append(row)
    return pd.DataFrame(rows).set_index("model")


rates = category_rates(summaries)
display(rates.round(1))

delta = pd.DataFrame({
    "Depth − RGB": rates.loc["Depth"] - rates.loc["RGB"],
    "Flow − RGB": rates.loc["Flow"] - rates.loc["RGB"],
    "MM − RGB": rates.loc["MM"] - rates.loc["RGB"],
})
display(delta.round(1))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
cats = ["Camera", "Background", "Noise", "Layout"]
x = range(len(cats))
w = 0.2
for i, m in enumerate(MODELS):
    ax.bar([xi + (i - 1.5) * w for xi in x], rates.loc[m, cats], width=w, label=m)
ax.set_xticks(list(x), cats)
ax.set_ylabel("Success rate (%)")
ax.set_title("LIBERO-Plus perturbation success by model")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Failure overlap (are failures common?)

In [ ]:
bucket_counts = tasks["bucket"].value_counts()
mixed = tasks[tasks["bucket"] == "mixed"]
print("Outcome buckets:")
print(f"  all_success: {(tasks['bucket']=='all_success').sum()} ({100*(tasks['bucket']=='all_success').mean():.1f}%)")
print(f"  all_fail:    {(tasks['bucket']=='all_fail').sum()} ({100*(tasks['bucket']=='all_fail').mean():.1f}%)")
print(f"  mixed:       {len(mixed)} ({100*len(mixed)/len(tasks):.1f}%)")

fail_sets = {m: set(tasks.loc[~tasks[f"success_{m}"], "task_id"]) for m in MODELS}
union_fail = set.union(*fail_sets.values())
common_fail = set.intersection(*fail_sets.values())
print(f"\nUnion failures (any model): {len(union_fail)}")
print(f"Intersection (all 4 fail): {len(common_fail)} ({100*len(common_fail)/max(len(union_fail),1):.1f}% of union)")

for m in MODELS:
    print(f"  {m} fails: {len(fail_sets[m])} ({100*len(fail_sets[m])/len(tasks):.1f}%)")

In [ ]:
overlap_rows = []
for a in MODELS:
    for b in MODELS:
        if a >= b:
            continue
        inter = len(fail_sets[a] & fail_sets[b])
        overlap_rows.append({
            "pair": f"{a} ∩ {b}",
            "common_fails": inter,
            "pct_of_a": 100 * inter / max(len(fail_sets[a]), 1),
            "pct_of_b": 100 * inter / max(len(fail_sets[b]), 1),
        })
pd.DataFrame(overlap_rows).round(1)

In [ ]:
all_fail = tasks[tasks["bucket"] == "all_fail"]
print(f"All-4-fail tasks: {len(all_fail)}\n")
print("By category:")
display(all_fail["category_short"].value_counts().to_frame("count"))
print("By suite:")
display(all_fail["suite"].value_counts().to_frame("count"))
print("By difficulty:")
display(all_fail["difficulty"].value_counts().sort_index().to_frame("count"))
print("Top descriptions:")
display(all_fail["description"].value_counts().head(15).to_frame("count"))

## 3. RGB vs Multimodal disagreements

In [ ]:
rgb_fail_mm_ok = tasks[(~tasks["success_RGB"]) & tasks["success_MM"]]
rgb_ok_mm_fail = tasks[tasks["success_RGB"] & (~tasks["success_MM"])]

print(f"RGB fail, MM success: {len(rgb_fail_mm_ok)}")
print(f"RGB success, MM fail: {len(rgb_ok_mm_fail)}")

for label, sub in [("RGB fail → MM ok", rgb_fail_mm_ok), ("RGB ok → MM fail", rgb_ok_mm_fail)]:
    print(f"\n{label} by category:")
    display(sub["category_short"].value_counts().to_frame("count"))

In [ ]:
def disagreement_table(model_a: str, model_b: str) -> pd.DataFrame:
    sa, sb = f"success_{model_a}", f"success_{model_b}"
    a_win = tasks[(~tasks[sa]) & tasks[sb]]
    b_win = tasks[tasks[sa] & (~tasks[sb])]
    ties = tasks[tasks[sa] == tasks[sb]]
    return pd.DataFrame([
        {"comparison": f"{model_b} wins", "count": len(a_win)},
        {"comparison": f"{model_a} wins", "count": len(b_win)},
        {"comparison": "same outcome", "count": len(ties)},
    ])

for cat in ["Camera", "Background", "Noise", "Layout"]:
    sub = tasks[tasks["category_short"] == cat]
    a_win = ((~sub["success_RGB"]) & sub["success_MM"]).sum()
    b_win = (sub["success_RGB"] & (~sub["success_MM"])).sum()
    ties = (sub["success_RGB"] == sub["success_MM"]).sum()
    print(f"{cat} (n={len(sub)}): MM wins {a_win}, RGB wins {b_win}, ties {ties}")

## 4. Model-only failures & export

In [ ]:
for m in MODELS:
    others = set.union(*(fail_sets[o] for o in MODELS if o != m))
    only = fail_sets[m] - others
    print(f"{m}-only failures: {len(only)}")

OUT = REPO / "notebooks" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

tasks.to_csv(OUT / "libero_plus_task_outcomes.csv", index=False)
all_fail.to_csv(OUT / "libero_plus_all_fail_tasks.csv", index=False)
rgb_fail_mm_ok.to_csv(OUT / "libero_plus_rgb_fail_mm_success.csv", index=False)
rates.round(2).to_csv(OUT / "libero_plus_category_rates.csv")

print(f"\nWrote CSVs to {OUT}")

In [ ]:
# Optional: inspect a specific task across models
QUERY = "libero_goal"  # suite prefix, category_short, or substring of description
mask = (
    tasks["task_id"].str.contains(QUERY, case=False)
    | tasks["description"].str.contains(QUERY, case=False, na=False)
    | tasks["category_short"].str.contains(QUERY, case=False, na=False)
)
cols = ["task_id", "category_short", "difficulty", "description"] + [f"success_{m}" for m in MODELS]
tasks.loc[mask, cols].head(20)